# 5 · Send the annotations to ELAN

**What this notebook does:** It turns the results of Notebooks 3a and 4 (and 3b, if you ran it) into **tiers** in
an ELAN file that is linked to your video. In ELAN you can then watch every automatically found gesture, check it,
and correct it.

| You need | You get |
|---|---|
| `Gestures/Test_Video_gestures_found.csv` (from Notebook 4) | `Elan/Test_Video.eaf`: an ELAN file with the tiers `gesture_id`, `peaks` and `min` (and `apex`). |
| Optional: `Apexes/Test_Video_apexes.csv` (from Notebook 3b) | `Annotations/Test_Video_annotations.csv`: the same annotations as a table, for Excel, R, SPSS, ... |

**Your own annotations are safe.** If `Elan/Test_Video.eaf` already exists, only the tiers made by this notebook
are replaced; all your other tiers (like `Stroke`) stay as they are. The previous version of the file is kept as
`Elan/Test_Video_backup.eaf`.

⚠️ **Close the file in ELAN before you run this notebook.** Otherwise ELAN may later save its open copy over ours.

## Step 1 · Load the tools

In [ ]:
from pathlib import Path
import mimetypes                 # works out the type of a video file (needed by ELAN)
import pandas as pd
import plotly.graph_objects as go
from pympi import Elan           # reads and writes ELAN (.eaf) files

## Step 2 · Settings

In [ ]:
VIDEO_NAME = "Test_Video"

# Columns from Notebook 4 that become ELAN tiers:
#   "gesture_id" = the gestures found in Notebook 3a, numbered 1, 2, 3, ...
#   "peaks"      = the stretch between the two biggest speed peaks of each gesture (Notebook 4)
#   "min"        = the slowest moment between those two peaks (Notebook 4)
TIERS = ["gesture_id", "peaks", "min"]

# Make every annotation this many milliseconds longer (one video frame), so that marks for a single
# moment (like "min" and "apex") are wide enough to see and click in ELAN.
EXTEND_END_MS = 33

## Step 3 · Turn the results into annotations

An ELAN annotation has a start time, an end time and a text. Our results have one value every 10 ms, so each
stretch of identical values becomes one annotation. For example, if gesture 3 runs from 7810 ms to 8550 ms, it
becomes one annotation with the text `3` on the `gesture_id` tier. Empty values and 0 ("no") are skipped.

In [ ]:
def to_annotations(data, column):
    """Turn a column with one value per 10 ms into a list of (start, end, text) annotations."""
    # Write every value as text (3.0 becomes "3").
    # Empty values and 0 ("no") become "", which means: nothing to annotate.
    texts = pd.Series(["" if pd.isna(value) or value == 0 else str(value).removesuffix(".0")
                       for value in data[column]], index=data.index)
    stretch_number = (texts != texts.shift()).cumsum()  # goes up by 1 every time the text changes

    annotations = []
    for _, stretch in texts.groupby(stretch_number):
        text = stretch.iloc[0]
        if text == "":
            continue
        start = int(data.loc[stretch.index[0], "time_ms"])
        end = int(data.loc[stretch.index[-1], "time_ms"]) + EXTEND_END_MS
        annotations.append((start, end, text))
    return annotations


gestures = pd.read_csv(f"../Gestures/{VIDEO_NAME}_gestures_found.csv")
tiers = {tier: to_annotations(gestures, tier) for tier in TIERS}

# Add the apexes from Notebook 3b, if that notebook was run
apex_path = Path(f"../Apexes/{VIDEO_NAME}_apexes.csv")
if apex_path.exists():
    tiers["apex"] = to_annotations(pd.read_csv(apex_path), "apex")

for tier, annotations in tiers.items():
    print(f"{tier}: {len(annotations)} annotations")

## Step 4 · Write the ELAN file

If the ELAN file already exists we open it, so your own tiers are kept. Otherwise we make a new one and link it
to the video.

In [ ]:
elan_path = Path(f"../Elan/{VIDEO_NAME}.eaf")
elan_path.parent.mkdir(exist_ok=True)

if elan_path.exists():
    elan = Elan.Eaf(str(elan_path))
else:
    elan = Elan.Eaf()
    elan.remove_tier("default")  # a new ELAN file starts with an empty tier we don't need

# Link the video, if the file isn't linked to one yet
if not elan.media_descriptors:
    video_path = next(Path("../Videos").glob(VIDEO_NAME + ".*"))
    elan.add_linked_file(video_path.resolve().as_uri(),
                         relpath=f"../Videos/{video_path.name}",
                         mimetype=mimetypes.guess_type(video_path.name)[0] or "video/mp4")

# Replace our tiers with the new results
for tier, annotations in tiers.items():
    if tier in elan.get_tier_names():
        elan.remove_tier(tier)
    elan.add_tier(tier)
    for start, end, text in annotations:
        elan.add_annotation(tier, start, end, text)

# Tell ELAN the highest annotation number in use,
# so that it won't reuse one when you add annotations by hand
elan.properties = [p for p in elan.properties if not p[0].startswith("lastUsedAnnotation")]
elan.properties.append(("lastUsedAnnotationId", elan.maxaid))

# Keep the previous version as a backup, then save
if elan_path.exists():
    elan_path.replace(elan_path.with_name(f"{VIDEO_NAME}_backup.eaf"))
Elan.to_eaf(str(elan_path), elan)
print("Saved:", elan_path)
print("Tiers in the file:", ", ".join(elan.get_tier_names()))

## Step 5 · Save the annotations as a table too

In [ ]:
rows = []
for tier, annotations in tiers.items():
    for start, end, text in annotations:
        rows.append((tier, start, end, text))
table = pd.DataFrame(rows, columns=["tier", "start_ms", "end_ms", "value"])

Path("../Annotations").mkdir(exist_ok=True)
table_path = f"../Annotations/{VIDEO_NAME}_annotations.csv"
table.to_csv(table_path, index=False)
print("Saved:", table_path)
table.head(10)

## Step 6 · Preview

A quick preview of the new tiers, laid out like ELAN's timeline. Hover over a bar to see its text and times,
and drag across the plot to zoom in.

In [ ]:
fig = go.Figure()
for tier, rows in table.groupby("tier", sort=False):
    fig.add_bar(y=rows["tier"], base=rows["start_ms"], x=rows["end_ms"] - rows["start_ms"],
                orientation="h", name=tier, text=rows["value"],
                customdata=rows[["start_ms", "end_ms"]],
                hovertemplate="%{text}<br>%{customdata[0]} - %{customdata[1]} ms<extra></extra>")
fig.update_layout(title="The new ELAN tiers", xaxis_title="Time (ms)", showlegend=False, height=300)
fig.update_yaxes(autorange="reversed")  # first tier on top, like in ELAN
fig.show()

## Step 7 · Open the result in ELAN

1. Start ELAN and open `Elan/Test_Video.eaf` (*File → Open…*).
2. If ELAN says it can't find the video, choose `Videos/Test_Video.MOV`.
3. Each tier is a row below the video. Click an annotation to jump to that moment, and press play to watch the gesture.

**You've finished the workflow!** To run it on your own video, see "Use your own videos" in the README.